In [0]:
events = spark.read.table("default.bronze_events")

from pyspark.sql import functions as F

label_df = events.groupBy("user_id") \
    .agg(
        F.max(
            F.when(F.col("event_type") == "purchase", 1).otherwise(0)
        ).alias("purchased")
    )

display(label_df)

In [0]:
features_df = spark.read.table("default.silver_user_features")

In [0]:
from pyspark.sql import functions as F

label_df = events.groupBy("user_id") \
    .agg(
        F.max(
            F.when(F.col("event_type") == "purchase", 1).otherwise(0)
        ).alias("purchased")
    )

training_data = features_df.join(label_df, "user_id", "left")

display(training_data)

In [0]:
training_data.groupBy("purchased").count().show()

In [0]:
from pyspark.sql import functions as F

# Calculate counts
class_counts = training_data.groupBy("purchased").count().collect()

count_dict = {row["purchased"]: row["count"] for row in class_counts}

total = sum(count_dict.values())

# Add weight column
training_data = training_data.withColumn(
    "class_weight",
    F.when(F.col("purchased") == 1,
           total / (2 * count_dict[1]))
     .otherwise(total / (2 * count_dict[0]))
)

training_data.groupBy("purchased").agg(
    F.avg("class_weight")
).show()

In [0]:
train, test = training_data.randomSplit([0.8, 0.2], seed=42)

print("Train:", train.count())
print("Test:", test.count())

train.groupBy("purchased").count().show()
test.groupBy("purchased").count().show()